In [1]:
# Retention KPI Development and Relationship Strength Analysis

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Set project paths and create output directories
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy"
)

INPUT_FILE = PROJECT_DIR / "data" / "processed" / "European_Bank_Cleaned_Analytical.csv"

TABLE_DIR = PROJECT_DIR / "outputs" / "tables" / "retention_kpi"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures" / "retention_kpi"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned analytical dataset not found:\n{INPUT_FILE}"
    )

# ------------------------------------------------------------
# Load dataset and standardize column names
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]", "", regex=True)
)

print("=" * 75)
print("RETENTION KPI DEVELOPMENT")
print("=" * 75)

print(f"\nDataset loaded successfully.")
print(f"Number of customers: {len(df):,}")
print(f"Number of columns: {len(df.columns):,}")

required_columns = [
    "customerid",
    "numofproducts",
    "isactivemember",
    "exited",
    "balance",
    "hascrcard"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing from the dataset: {missing_columns}"
    )

if df.empty:
    raise ValueError("The dataset contains no customer records.")

if df["customerid"].duplicated().any():
    raise ValueError(
        "Duplicate Customer IDs detected. Resolve duplicates before calculating KPIs."
    )

# Convert required numeric columns safely
numeric_columns = [
    "numofproducts",
    "isactivemember",
    "exited",
    "balance",
    "hascrcard"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

if df[numeric_columns].isna().any().any():
    raise ValueError(
        "Missing or non-numeric values detected in required KPI columns."
    )

# Validate binary columns
for column in ["isactivemember", "exited", "hascrcard"]:
    if not set(df[column].unique()).issubset({0, 1}):
        raise ValueError(
            f"Unexpected values found in {column}. Expected only 0 and 1."
        )

if (df["numofproducts"] < 1).any():
    raise ValueError("NumOfProducts must be at least 1.")

if (df["balance"] < 0).any():
    raise ValueError("Negative customer balances detected.")

# ------------------------------------------------------------
# Define business KPI calculation rules
# ------------------------------------------------------------

KPI_DEFINITIONS = pd.DataFrame([
    {
        "kpi": "Overall Churn Rate",
        "formula": "Exited Customers / Total Customers * 100",
        "unit": "Percentage",
        "interpretation": "Percentage of customers who exited the bank."
    },
    {
        "kpi": "Overall Retention Rate",
        "formula": "Retained Customers / Total Customers * 100",
        "unit": "Percentage",
        "interpretation": "Percentage of customers who remained with the bank."
    },
    {
        "kpi": "Engagement Retention Ratio",
        "formula": "Active Retention Rate / Inactive Retention Rate",
        "unit": "Ratio",
        "interpretation": "Relative retention performance of active versus inactive customers."
    },
    {
        "kpi": "Product Depth Index",
        "formula": "Total Products Held / Total Customers",
        "unit": "Average products per customer",
        "interpretation": "Average number of products held by each customer."
    },
    {
        "kpi": "High-Balance Disengagement Rate",
        "formula": "Inactive High-Balance Customers / All High-Balance Customers * 100",
        "unit": "Percentage",
        "interpretation": "Percentage of high-balance customers classified as inactive."
    },
    {
        "kpi": "Credit Card Stickiness Score",
        "formula": "Cardholder Retention Rate - Non-Cardholder Retention Rate",
        "unit": "Percentage points",
        "interpretation": "Retention difference between cardholders and non-cardholders."
    },
    {
        "kpi": "Relationship Strength Index",
        "formula": "60 * Activity + 40 * Product Adoption",
        "unit": "Score from 0 to 100",
        "interpretation": "Proposed customer relationship strength measure."
    }
])

KPI_DEFINITIONS.to_csv(
    TABLE_DIR / "kpi_calculation_definitions.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Calculate overall customer retention and churn
# ------------------------------------------------------------

total_customers = len(df)

retained_customers = int((df["exited"] == 0).sum())

exited_customers = int((df["exited"] == 1).sum())

overall_churn_rate = (
    exited_customers / total_customers * 100
)

overall_retention_rate = (
    retained_customers / total_customers * 100
)

overall_customer_summary = pd.DataFrame([{
    "total_customers": total_customers,
    "retained_customers": retained_customers,
    "exited_customers": exited_customers,
    "overall_churn_rate_pct": overall_churn_rate,
    "overall_retention_rate_pct": overall_retention_rate
}])

overall_customer_summary.to_csv(
    TABLE_DIR / "overall_customer_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Calculate engagement retention ratio
# ------------------------------------------------------------

engagement_summary = (
    df.groupby("isactivemember")
    .agg(
        total_customers=("customerid", "count"),
        retained_customers=("exited", lambda x: (x == 0).sum()),
        exited_customers=("exited", lambda x: (x == 1).sum())
    )
    .reset_index()
)

engagement_summary["activity_status"] = (
    engagement_summary["isactivemember"]
    .map({1: "Active", 0: "Inactive"})
)

engagement_summary["retention_rate_pct"] = (
    engagement_summary["retained_customers"]
    / engagement_summary["total_customers"] * 100
)

engagement_summary["churn_rate_pct"] = (
    engagement_summary["exited_customers"]
    / engagement_summary["total_customers"] * 100
)

active_rows = engagement_summary[
    engagement_summary["isactivemember"] == 1
]

inactive_rows = engagement_summary[
    engagement_summary["isactivemember"] == 0
]

active_retention_rate = (
    active_rows["retention_rate_pct"].iloc[0]
    if not active_rows.empty else np.nan
)

inactive_retention_rate = (
    inactive_rows["retention_rate_pct"].iloc[0]
    if not inactive_rows.empty else np.nan
)

engagement_retention_ratio = (
    active_retention_rate / inactive_retention_rate
    if pd.notna(active_retention_rate)
    and pd.notna(inactive_retention_rate)
    and inactive_retention_rate != 0
    else np.nan
)

engagement_summary.to_csv(
    TABLE_DIR / "engagement_retention_kpi.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Calculate Product Depth Index
# ------------------------------------------------------------

total_products_held = df["numofproducts"].sum()

product_depth_index = (
    total_products_held / total_customers
)

product_summary = (
    df.groupby("numofproducts")
    .agg(
        total_customers=("customerid", "count"),
        retained_customers=("exited", lambda x: (x == 0).sum()),
        exited_customers=("exited", lambda x: (x == 1).sum())
    )
    .reset_index()
    .sort_values("numofproducts")
)

product_summary["retention_rate_pct"] = (
    product_summary["retained_customers"]
    / product_summary["total_customers"] * 100
)

product_summary["churn_rate_pct"] = (
    product_summary["exited_customers"]
    / product_summary["total_customers"] * 100
)

product_summary.to_csv(
    TABLE_DIR / "product_depth_and_retention.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Calculate High-Balance Disengagement Rate
# ------------------------------------------------------------

positive_balances = df.loc[df["balance"] > 0, "balance"]

if positive_balances.empty:
    high_balance_threshold = np.nan
    df["high_balance_flag"] = 0
else:
    high_balance_threshold = positive_balances.quantile(0.75)

    df["high_balance_flag"] = (
        df["balance"] >= high_balance_threshold
    ).astype(int)

df["inactive_high_balance_flag"] = (
    (df["high_balance_flag"] == 1)
    & (df["isactivemember"] == 0)
).astype(int)

high_balance_customers = df[
    df["high_balance_flag"] == 1
]

total_high_balance_customers = len(high_balance_customers)

inactive_high_balance_customers = int(
    high_balance_customers["inactive_high_balance_flag"].sum()
)

high_balance_disengagement_rate = (
    inactive_high_balance_customers
    / total_high_balance_customers * 100
    if total_high_balance_customers > 0
    else np.nan
)

high_balance_retained_inactive = int(
    (
        (high_balance_customers["isactivemember"] == 0)
        & (high_balance_customers["exited"] == 0)
    ).sum()
)

high_balance_exited = int(
    (high_balance_customers["exited"] == 1).sum()
)

high_balance_summary = pd.DataFrame([{
    "high_balance_threshold": high_balance_threshold,
    "total_high_balance_customers": total_high_balance_customers,
    "inactive_high_balance_customers": inactive_high_balance_customers,
    "retained_inactive_high_balance_customers": high_balance_retained_inactive,
    "exited_high_balance_customers": high_balance_exited,
    "high_balance_disengagement_rate_pct": high_balance_disengagement_rate
}])

high_balance_summary.to_csv(
    TABLE_DIR / "high_balance_disengagement_kpi.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Calculate Credit Card Stickiness Score
# ------------------------------------------------------------

card_summary = (
    df.groupby("hascrcard")
    .agg(
        total_customers=("customerid", "count"),
        retained_customers=("exited", lambda x: (x == 0).sum()),
        exited_customers=("exited", lambda x: (x == 1).sum())
    )
    .reset_index()
)

card_summary["cardholder_status"] = (
    card_summary["hascrcard"]
    .map({1: "Cardholder", 0: "Non-Cardholder"})
)

card_summary["retention_rate_pct"] = (
    card_summary["retained_customers"]
    / card_summary["total_customers"] * 100
)

card_summary["churn_rate_pct"] = (
    card_summary["exited_customers"]
    / card_summary["total_customers"] * 100
)

cardholder_rows = card_summary[
    card_summary["hascrcard"] == 1
]

non_cardholder_rows = card_summary[
    card_summary["hascrcard"] == 0
]

cardholder_retention_rate = (
    cardholder_rows["retention_rate_pct"].iloc[0]
    if not cardholder_rows.empty else np.nan
)

non_cardholder_retention_rate = (
    non_cardholder_rows["retention_rate_pct"].iloc[0]
    if not non_cardholder_rows.empty else np.nan
)

credit_card_stickiness_score = (
    cardholder_retention_rate - non_cardholder_retention_rate
    if pd.notna(cardholder_retention_rate)
    and pd.notna(non_cardholder_retention_rate)
    else np.nan
)

card_summary.to_csv(
    TABLE_DIR / "credit_card_stickiness_kpi.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Calculate Relationship Strength Index
# ------------------------------------------------------------

df["activity_score"] = (
    df["isactivemember"] * 1.0
)

df["product_adoption_score"] = (
    np.minimum(df["numofproducts"], 2) / 2
)

df["relationship_strength_index"] = (
    60 * df["activity_score"]
    + 40 * df["product_adoption_score"]
)

# ------------------------------------------------------------
# Create relationship strength tiers
# ------------------------------------------------------------

rsi_bins = [-0.01, 24, 49, 74, 89, 100]

rsi_labels = [
    "Very Weak",
    "Weak",
    "Moderate",
    "Strong",
    "Very Strong"
]

df["relationship_strength_category"] = pd.cut(
    df["relationship_strength_index"],
    bins=rsi_bins,
    labels=rsi_labels,
    include_lowest=True
)

# ------------------------------------------------------------
# Calculate churn and retention by relationship tier
# ------------------------------------------------------------

rsi_tier_summary = (
    df.groupby(
        "relationship_strength_category",
        observed=False
    )
    .agg(
        total_customers=("customerid", "count"),
        retained_customers=("exited", lambda x: (x == 0).sum()),
        exited_customers=("exited", lambda x: (x == 1).sum()),
        average_rsi=("relationship_strength_index", "mean"),
        average_products=("numofproducts", "mean"),
        active_customers=("isactivemember", "sum")
    )
    .reset_index()
)

rsi_tier_summary["retention_rate_pct"] = (
    rsi_tier_summary["retained_customers"]
    / rsi_tier_summary["total_customers"].replace(0, np.nan)
    * 100
)

rsi_tier_summary["churn_rate_pct"] = (
    rsi_tier_summary["exited_customers"]
    / rsi_tier_summary["total_customers"].replace(0, np.nan)
    * 100
)

rsi_tier_summary["customer_share_pct"] = (
    rsi_tier_summary["total_customers"]
    / total_customers * 100
)

rsi_tier_summary.to_csv(
    TABLE_DIR / "relationship_strength_tier_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Compare relationship strength across customer segments
# ------------------------------------------------------------

segment_columns = [
    "geography",
    "gender",
    "agegroup",
    "tenuregroup",
    "balancecategory",
    "productcountcategory",
    "activitystatus"
]

available_segment_columns = [
    column for column in segment_columns
    if column in df.columns
]

segment_results = []

for segment_column in available_segment_columns:

    segment_data = df.copy()

    segment_data[segment_column] = (
        segment_data[segment_column]
        .fillna("Unknown")
        .astype(str)
    )

    grouped = (
        segment_data.groupby(segment_column)
        .agg(
            total_customers=("customerid", "count"),
            retained_customers=("exited", lambda x: (x == 0).sum()),
            exited_customers=("exited", lambda x: (x == 1).sum()),
            average_rsi=("relationship_strength_index", "mean"),
            average_products=("numofproducts", "mean"),
            active_customers=("isactivemember", "sum")
        )
        .reset_index()
    )

    grouped["retention_rate_pct"] = (
        grouped["retained_customers"]
        / grouped["total_customers"] * 100
    )

    grouped["churn_rate_pct"] = (
        grouped["exited_customers"]
        / grouped["total_customers"] * 100
    )

    grouped["customer_share_pct"] = (
        grouped["total_customers"] / total_customers * 100
    )

    grouped.insert(0, "segment_dimension", segment_column)

    grouped = grouped.rename(
        columns={segment_column: "segment_value"}
    )

    segment_results.append(grouped)

if segment_results:
    rsi_segment_comparison = pd.concat(
        segment_results,
        ignore_index=True
    )
else:
    rsi_segment_comparison = pd.DataFrame(
        columns=[
            "segment_dimension",
            "segment_value",
            "total_customers",
            "retained_customers",
            "exited_customers",
            "average_rsi",
            "average_products",
            "active_customers",
            "retention_rate_pct",
            "churn_rate_pct",
            "customer_share_pct"
        ]
    )

rsi_segment_comparison.to_csv(
    TABLE_DIR / "relationship_strength_segment_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Prepare consolidated KPI summary
# ------------------------------------------------------------

def safe_round(value, digits=4):
    if pd.isna(value):
        return np.nan
    return round(float(value), digits)


kpi_summary = pd.DataFrame([
    {
        "kpi_name": "Total Customers",
        "kpi_value": total_customers,
        "unit": "Customers",
        "description": "Total number of unique customers."
    },
    {
        "kpi_name": "Retained Customers",
        "kpi_value": retained_customers,
        "unit": "Customers",
        "description": "Number of customers who have not exited."
    },
    {
        "kpi_name": "Exited Customers",
        "kpi_value": exited_customers,
        "unit": "Customers",
        "description": "Number of customers who have exited."
    },
    {
        "kpi_name": "Overall Churn Rate",
        "kpi_value": safe_round(overall_churn_rate),
        "unit": "Percentage",
        "description": "Percentage of customers who exited."
    },
    {
        "kpi_name": "Overall Retention Rate",
        "kpi_value": safe_round(overall_retention_rate),
        "unit": "Percentage",
        "description": "Percentage of customers retained."
    },
    {
        "kpi_name": "Engagement Retention Ratio",
        "kpi_value": safe_round(engagement_retention_ratio),
        "unit": "Ratio",
        "description": "Active retention rate divided by inactive retention rate."
    },
    {
        "kpi_name": "Product Depth Index",
        "kpi_value": safe_round(product_depth_index),
        "unit": "Average products",
        "description": "Average number of products per customer."
    },
    {
        "kpi_name": "High-Balance Disengagement Rate",
        "kpi_value": safe_round(high_balance_disengagement_rate),
        "unit": "Percentage",
        "description": "Inactive customers as a percentage of all high-balance customers."
    },
    {
        "kpi_name": "Credit Card Stickiness Score",
        "kpi_value": safe_round(credit_card_stickiness_score),
        "unit": "Percentage points",
        "description": "Cardholder retention rate minus non-cardholder retention rate."
    },
    {
        "kpi_name": "Average Relationship Strength Index",
        "kpi_value": safe_round(
            df["relationship_strength_index"].mean()
        ),
        "unit": "Score out of 100",
        "description": "Average proposed relationship strength score."
    }
])

kpi_summary.to_csv(
    TABLE_DIR / "consolidated_kpi_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# Dashboard-friendly KPI table
dashboard_kpi_summary = kpi_summary.copy()

dashboard_kpi_summary["kpi_value"] = pd.to_numeric(
    dashboard_kpi_summary["kpi_value"],
    errors="coerce"
)

dashboard_kpi_summary.to_csv(
    TABLE_DIR / "dashboard_kpi_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Prepare customer-level retention KPI dataset
# ------------------------------------------------------------

customer_kpi_columns = [
    "customerid",
    "isactivemember",
    "exited",
    "numofproducts",
    "balance",
    "hascrcard",
    "high_balance_flag",
    "inactive_high_balance_flag",
    "activity_score",
    "product_adoption_score",
    "relationship_strength_index",
    "relationship_strength_category"
]

for optional_column in [
    "geography",
    "gender",
    "age",
    "agegroup",
    "tenure",
    "tenuregroup",
    "balancecategory",
    "productcountcategory"
]:
    if optional_column in df.columns:
        customer_kpi_columns.append(optional_column)

customer_kpi_dataset = df[customer_kpi_columns].copy()

customer_kpi_dataset.to_csv(
    TABLE_DIR / "customer_retention_kpi_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Run KPI validation checks
# ------------------------------------------------------------

validation_results = []

def record_check(check_name, condition, details):
    validation_results.append({
        "validation_check": check_name,
        "status": "PASSED" if condition else "FAILED",
        "details": details
    })

record_check(
    "Total customer reconciliation",
    total_customers == retained_customers + exited_customers,
    f"{total_customers} = {retained_customers} + {exited_customers}"
)

record_check(
    "Customer ID uniqueness",
    not df["customerid"].duplicated().any(),
    "Every customer ID is unique."
)

record_check(
    "Churn and retention complement",
    np.isclose(
        overall_churn_rate + overall_retention_rate,
        100
    ),
    f"Combined rate: {overall_churn_rate + overall_retention_rate:.4f}%"
)

record_check(
    "Engagement customer reconciliation",
    engagement_summary["total_customers"].sum() == total_customers,
    "Active and inactive customer counts reconcile with total customers."
)

record_check(
    "Product customer reconciliation",
    product_summary["total_customers"].sum() == total_customers,
    "Product count groups reconcile with total customers."
)

record_check(
    "Credit card customer reconciliation",
    card_summary["total_customers"].sum() == total_customers,
    "Cardholder and non-cardholder counts reconcile with total customers."
)

record_check(
    "High-balance customer reconciliation",
    total_high_balance_customers
    == len(high_balance_customers),
    "High-balance customer count matches the filtered dataset."
)

record_check(
    "High-balance disengagement rate bounds",
    pd.isna(high_balance_disengagement_rate)
    or 0 <= high_balance_disengagement_rate <= 100,
    "High-balance disengagement rate is between 0% and 100%, when defined."
)

record_check(
    "Relationship Strength Index range",
    df["relationship_strength_index"].between(0, 100).all(),
    "All RSI scores are between 0 and 100."
)

record_check(
    "Relationship tier assignment",
    df["relationship_strength_category"].notna().all(),
    "Every customer has an assigned relationship strength tier."
)

record_check(
    "RSI tier customer reconciliation",
    rsi_tier_summary["total_customers"].sum() == total_customers,
    "Relationship tier counts reconcile with total customers."
)

record_check(
    "RSI tier churn and retention reconciliation",
    (
        rsi_tier_summary["retained_customers"].sum()
        + rsi_tier_summary["exited_customers"].sum()
    ) == total_customers,
    "Retained and exited customers across RSI tiers reconcile."
)

validation_df = pd.DataFrame(validation_results)

validation_df.to_csv(
    TABLE_DIR / "retention_kpi_validation_report.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# Display KPI results and validation report
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("OVERALL CUSTOMER RETENTION SUMMARY")
print("=" * 75)

print(overall_customer_summary.to_string(index=False))

print("\n" + "=" * 75)
print("CORE RETENTION KPIs")
print("=" * 75)

print(kpi_summary.to_string(index=False))

print("\n" + "=" * 75)
print("ENGAGEMENT RETENTION ANALYSIS")
print("=" * 75)

print(engagement_summary.to_string(index=False))

print("\n" + "=" * 75)
print("PRODUCT DEPTH AND RETENTION ANALYSIS")
print("=" * 75)

print(product_summary.to_string(index=False))

print("\n" + "=" * 75)
print("HIGH-BALANCE DISENGAGEMENT ANALYSIS")
print("=" * 75)

print(high_balance_summary.to_string(index=False))

print("\n" + "=" * 75)
print("CREDIT CARD STICKINESS ANALYSIS")
print("=" * 75)

print(card_summary.to_string(index=False))

print("\n" + "=" * 75)
print("RELATIONSHIP STRENGTH TIER ANALYSIS")
print("=" * 75)

print(rsi_tier_summary.to_string(index=False))

print("\n" + "=" * 75)
print("RELATIONSHIP STRENGTH SEGMENT COMPARISON")
print("=" * 75)

if not rsi_segment_comparison.empty:
    print(rsi_segment_comparison.to_string(index=False))
else:
    print("No optional customer segment columns were available.")

print("\n" + "=" * 75)
print("VALIDATION RESULTS")
print("=" * 75)

print(validation_df.to_string(index=False))

failed_checks = validation_df[
    validation_df["status"] == "FAILED"
]

if not failed_checks.empty:
    raise AssertionError(
        "One or more validation checks failed. Review the validation report."
    )

print("\nAll validation checks passed successfully.")

print("\n" + "=" * 75)
print("OUTPUT FILES")
print("=" * 75)

for output_file in sorted(TABLE_DIR.glob("*.csv")):
    print(output_file.name)

print("\nRetention KPI development completed successfully.")
print(f"Tables saved to: {TABLE_DIR}")
print(f"Figures directory: {FIGURE_DIR}")

RETENTION KPI DEVELOPMENT

Dataset loaded successfully.
Number of customers: 10,000
Number of columns: 26

OVERALL CUSTOMER RETENTION SUMMARY
 total_customers  retained_customers  exited_customers  overall_churn_rate_pct  overall_retention_rate_pct
           10000                7963              2037                   20.37                       79.63

CORE RETENTION KPIs
                           kpi_name  kpi_value              unit                                                       description
                    Total Customers 10000.0000         Customers                                 Total number of unique customers.
                 Retained Customers  7963.0000         Customers                          Number of customers who have not exited.
                   Exited Customers  2037.0000         Customers                              Number of customers who have exited.
                 Overall Churn Rate    20.3700        Percentage                               Perc